# Enterprise Text-to-SQL Agent

This notebook documents the assessment prototype at a reproducible, presentation-ready level. The production implementation lives in `text_to_sql_agent_mvp.py`; the notebook imports that module rather than duplicating the full backend code.

The system translates natural-language questions into safe SQLite `SELECT` queries using schema-only retrieval augmented generation (RAG), an LLM backend, and deterministic SQL safety checks.

## Assessment Framing

**Problem.** Non-technical users often need answers from relational databases but cannot write SQL safely.

**AI approach.** The project combines structural knowledge representation (database schema), schema retrieval, LLM-based semantic parsing, and logic-based safety validation.

**Evaluation.** Benchmark cases compare generated SQL results against gold SQL results across university, retail, and healthcare demo databases.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

import text_to_sql_agent_mvp as agent

agent.load_env()
print("Backend loaded")

## Demo Databases

The repository includes three small SQLite databases for repeatable testing:

- `university_agent.db`: students, courses, grades
- `retail_analytics.db`: customers, products, orders, returns, support tickets
- `healthcare_analytics.db`: patients, doctors, hospitals, appointments, treatments

In [ ]:
demo_dbs = {
    "university": Path("data/university_agent.db"),
    "retail": Path("data/retail_analytics.db"),
    "healthcare": Path("data/healthcare_analytics.db"),
}

for name, path in demo_dbs.items():
    print(f"{name}: {path} exists={path.exists()}")

## Schema Knowledge Representation

The database schema is used as a structured representation of domain knowledge. Tables represent entities, columns represent attributes, and foreign keys represent relationships. The LLM receives schema metadata, not raw row data.

In [ ]:
schema = agent.get_schema(str(demo_dbs["university"]))
print(schema[:1200])

## Schema RAG Demonstration

Schema RAG retrieves only the most relevant table definitions for a user question. This reduces prompt size and limits unnecessary metadata exposure.

In [ ]:
question = "What is the average score for each course?"
context = agent.retrieve_schema_context(str(demo_dbs["university"]), question, top_k=3)
print(context.report)
print("\nRetrieved schema:\n")
print(context.schema_text)

## SQL Safety Layer

The LLM is not allowed to execute SQL directly. Python validates generated SQL before execution and opens SQLite in read-only mode.

In [ ]:
examples = [
    "SELECT full_name FROM students LIMIT 5;",
    "DROP TABLE students;",
    "SELECT * FROM sqlite_master;",
]

for sql in examples:
    print(f"{sql} -> safe={agent.is_safe_query(sql)}")

## End-to-End Query Example

This cell patches the LLM call with a known SQL query, so the workflow is runnable without consuming API quota. For live Gemini or Ollama tests, use the evaluation script below.

In [ ]:
from unittest.mock import patch

with patch.object(agent, "generate_sql", return_value="SELECT major, COUNT(*) AS student_count FROM students GROUP BY major ORDER BY major"):
    result = agent.ask_database(
        "How many students are enrolled in each major?",
        db_path=str(demo_dbs["university"]),
    )

pd.DataFrame(result.rows, columns=result.columns)

## Automatic Evaluation

The benchmark file `evaluation/cases.json` contains natural-language questions, gold SQL, expected tables, and difficulty levels. Metrics include safe SQL rate, execution success, row match, exact result match, schema table recall, and latency.

In [ ]:
cases = json.loads(Path("evaluation/cases.json").read_text(encoding="utf-8"))
pd.DataFrame(cases)[["id", "dataset", "difficulty", "question"]]

### Gold SQL Baseline

The gold baseline validates that all benchmark SQL queries execute successfully against the three demo databases.

In [ ]:
# Run from a terminal for full CSV/Markdown output:
# python scripts/evaluate_text_to_sql.py --mode gold

summary = []
for case in cases:
    result = agent.execute_query(case["db_path"], case["gold_sql"])
    summary.append({
        "case": case["id"],
        "dataset": case["dataset"],
        "safe_sql": agent.is_safe_query(case["gold_sql"]),
        "executed": result.error is None,
        "rows": len(result.rows),
    })

pd.DataFrame(summary)

### LLM Evaluation Commands

Use these commands to evaluate real model performance. Add delay for Gemini free-tier quota limits.

In [ ]:
# Gemini hosted model
# python scripts/evaluate_text_to_sql.py --mode llm --provider gemini --model gemini-2.5-flash --delay-seconds 15

# Local Ollama model
# ollama pull gemma3
# python scripts/evaluate_text_to_sql.py --mode llm --provider ollama --model gemma3

## Critical Reflection

Strengths:

- Schema-only prompting reduces data exposure.
- Deterministic safety checks prevent write operations.
- The generated SQL and retrieval report support transparency.
- The benchmark gives reproducible empirical evidence.

Limitations:

- LLMs can still hallucinate columns or incorrect joins.
- Exact-result evaluation is strict and may penalise semantically equivalent SQL.
- Local Ollama deployment depends on user hardware and cannot be hosted directly on Streamlit Community Cloud.
- Larger enterprise schemas may require embedding retrieval or database-specific metadata governance.

## References

- Lewis et al. (2020), Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.
- Yu et al. (2018), Spider: A Large-Scale Human-Labeled Dataset for Complex and Cross-Domain Semantic Parsing and Text-to-SQL.
- Zhong et al. (2017), Seq2SQL: Generating Structured Queries from Natural Language using Reinforcement Learning.
- NIST (2023), Artificial Intelligence Risk Management Framework.